### Building the call library from an annotated subsample csv
Runs locally on Mac, reads mounted RDS volumes, extracts 0.5s peak-energy,
audio segments resampled to 256 kHz, displays high-res Mel Spectrograms, 
and exports cleanly named WAV clips.


We used a csv file that annotated a list of bat species that were not my target bat and clipped them, one per file.

In [ ]:
# previous clipping cell for noise - non target bat calls from a csv annotated with non target bat species
import os
import sys
import re
import numpy as np
import pandas as pd
import librosa
import librosa.display
import soundfile as sf
import matplotlib.pyplot as plt
CSV_ANNOTATIONS = '/Users/isabellesecord/Library/CloudStorage/OneDrive-ImperialCollegeLondon/AA - MRES PROJECT/Model Training Version 6/Call_Library/Bat species classification verified list.csv'
OUTPUT_DIR = '/Users/isabellesecord/Library/CloudStorage/OneDrive-ImperialCollegeLondon/AA - MRES PROJECT/Model Training Version 6/Call_Library/0 - negative target'

# Mapping to specific folders based on serial numbers / folder names of recorder types
SERIAL_TO_FOLDER_MAP = {
    "SMU13265": "/Volumes/AcousticData/AcousticData_2024/AcousticData_Turmalina_2024/BatData_Turmalina_2024/SM_05/Data",
    "SM_05":    "/Volumes/AcousticData/AcousticData_2024/AcousticData_Turmalina_2024/BatData_Turmalina_2024/SM_05/Data",
    "SMU13587": "/Volumes/AcousticData/AcousticData_2024/AcousticData_Turmalina_2024/BatData_Turmalina_2024/SM_07/Data",
    "SM_07":    "/Volumes/AcousticData/AcousticData_2024/AcousticData_Turmalina_2024/BatData_Turmalina_2024/SM_07/Data",
    "SMU13591": "/Volumes/AcousticData/AcousticData_2024/AcousticData_Turmalina_2024/BatData_Turmalina_2024/SM_02/Data",
    "SM_02":    "/Volumes/AcousticData/AcousticData_2024/AcousticData_Turmalina_2024/BatData_Turmalina_2024/SM_02/Data",
    "SMU13372": "/Volumes/AcousticData/AcousticData_2024/AcousticData_Turmalina_2024/BatData_Turmalina_2024/SM_16/Data",
    "SM_16":    "/Volumes/AcousticData/AcousticData_2024/AcousticData_Turmalina_2024/BatData_Turmalina_2024/SM_16/Data",
    "AM01":     "/Volumes/AcousticData/AcousticData_2023/Mombak_Turmalina_2023/BatData_Turmalina_2023/AM_01"
}

# Sets target sample rate and clip duration for 0.5s audio segments
TARGET_SR = 256000
CLIP_DURATION = 0.5
TARGET_SAMPLES = int(TARGET_SR * CLIP_DURATION)  # 128,000 samples

os.makedirs(OUTPUT_DIR, exist_ok=True)
def find_audio_file(filename):
    """Searches mounted RDS directories for the target filename."""
    # First search mapped directories
    for folder_path in set(SERIAL_TO_FOLDER_MAP.values()):
        if os.path.exists(folder_path):
            for root, _, files in os.walk(folder_path):
                if filename in files:
                    return os.path.join(root, filename)
                    
    # Fallback search across entire AcousticData mount if connected
    fallback_root = "/Volumes/AcousticData/"
    if os.path.exists(fallback_root):
        for root, _, files in os.walk(fallback_root):
            if filename in files:
                return os.path.join(root, filename)
                
    return None

#Finds the peak RMS energy in the audio and extracts a 0.5s clip centered around it
def extract_peak_energy_clip(y, sr, clip_samples=128000):
    """Locates max RMS energy and centers a 0.5s slice around it."""
    hop = 512
    rms = librosa.feature.rms(y=y, frame_length=2048, hop_length=hop)[0]
    
    peak_frame = np.argmax(rms)
    peak_sample = peak_frame * hop
    
    half_clip = clip_samples // 2
    start_sample = max(0, peak_sample - half_clip)
    end_sample = start_sample + clip_samples
    
    if end_sample > len(y):
        end_sample = len(y)
        start_sample = max(0, end_sample - clip_samples)
        
    segment = y[start_sample:end_sample]
    
    if len(segment) < clip_samples:
        segment = np.pad(segment, (0, clip_samples - len(segment)), mode='constant')
        
    return segment, start_sample / sr, end_sample / sr

# Displays a high-resolution Mel Spectrogram for visual verification of the extracted audio segment
def display_spectrogram(segment, sr, species_name, original_filename):
    """Plots a 256 Mel bin Spectrogram for visual verification."""
    plt.ion()  # Turn on interactive plotting
    fig, ax = plt.subplots(figsize=(10, 4.5))
    
    # 256 Mel Bins for high ultrasonic resolution
    S = librosa.feature.melspectrogram(
        y=segment, sr=sr, n_fft=2048, hop_length=512, 
        n_mels=256, fmin=20000, fmax=100000
    )
    S_db = librosa.power_to_db(S, ref=np.max)
    
    img = librosa.display.specshow(
        S_db, sr=sr, hop_length=512, x_axis='time', y_axis='mel',
        fmin=20000, fmax=100000, cmap='magma', ax=ax
    )
    
    fig.colorbar(img, ax=ax, format='%+2.0f dB')
    ax.set_title(f"Species / Annotation: {species_name}\nSource File: {original_filename}", fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    plt.pause(0.1)

 # This clips from the CSV annotations   
def run_clipper():
    if not os.path.exists(CSV_ANNOTATIONS):
        print(f"❌ Could not find CSV file at: {CSV_ANNOTATIONS}")
        print("💡 Remember to export your .numbers file to .csv first!") # Because macbook default software is numbers not excel
        sys.exit(1)
        
    df = pd.read_csv(CSV_ANNOTATIONS)
    print(f"📋 Loaded {len(df)} rows from CSV.")
    print("Columns found:", df.columns.tolist())
    
    # Assume Column 1 = File Name, Column 2 = Species/Class, reads the csv file and extracts the first two columns for processing
    col_file = df.columns[0]
    col_species = df.columns[1]
    
    for idx, row in df.iterrows():
        fname = str(row[col_file]).strip()
        species = str(row[col_species]).strip().replace(" ", "_")
        
        # Clean up species name string for filename
        species_clean = re.sub(r'[^\w\-]', '_', species)
        
        if not fname.lower().endswith('.wav'):
            fname += ".wav"
            
        print(f"\n────────────────────────────────────────────────────────")
        print(f"📌 [{idx+1}/{len(df)}] Processing: {fname} | Label: {species_clean}")
        
        full_path = find_audio_file(fname)
        if not full_path:
            print(f"⚠️ File NOT found in mounted volumes. Check volume connection.")
            continue
            
        print(f"📁 Located at: {full_path}")
        
        try:
            # Read audio & resample to 256 kHz
            y, native_sr = sf.read(full_path)
            if native_sr != TARGET_SR:
                print(f"🔄 Resampling {native_sr} Hz -> {TARGET_SR} Hz...")
                y = librosa.resample(y, orig_sr=native_sr, target_sr=TARGET_SR)
                
            # Extract 0.5s burst
            segment, start_t, end_t = extract_peak_energy_clip(y, TARGET_SR, TARGET_SAMPLES)
            
            # Show Mel Spectrogram
            display_spectrogram(segment, TARGET_SR, species_clean, fname)
            
            # Extract recorder name / metadata from folder path or filename
            recorder_id = "UNKNOWN_RECORDER"
            for key in SERIAL_TO_FOLDER_MAP.keys():
                if key in full_path:
                    recorder_id = key
                    break
                    
            # Parse date/time string from original filename (e.g., 20240315_213000)
            date_match = re.search(r'\d{8}_\d{6}', fname)
            date_str = date_match.group(0) if date_match else "NODATE"
            
            # Construct output filename: recordername_date_time_species.wav
            out_fname = f"{recorder_id}_{date_str}_{species_clean}.wav"
            out_path = os.path.join(OUTPUT_DIR, out_fname)
            
            # Save 256 kHz 0.5s audio clip
            sf.write(out_path, segment, TARGET_SR)
            print(f"💾 SAVED 0.5s CLIP: {out_fname}")
            
            # Prompt user to press enter or type 'skip'
            user_input = input("Press [Enter] for next file (or 'q' to quit): ")
            plt.close('all')
            if user_input.lower() == 'q':
                print("🛑 Stopping process early.")
                break
                
        except Exception as e:
            print(f"❌ Error processing {fname}: {e}")

if __name__ == "__main__":
    run_clipper()


In [ ]:
# interactive noise clipper from a positive list of saccopteryx calls csv - same again,  but with the positive calls from the csv file
"""
---------------------------
Fast Interactive 0.5s audio clipper for macOS over Network Drives.
- Direct path construction using 'Site' column to skip network os.walk freezes.
- Resamples audio on the fly to 256 kHz (128,000 samples).
- Displays high-res 256 Mel-bin Spectrograms (20-100 kHz).
- Controls: [y] Save, [n] Next Peak, [s] Skip, [p] Pause, [q] Quit.
"""

import os
import sys
import re
import numpy as np
import pandas as pd
import librosa
import librosa.display
import soundfile as sf
import matplotlib.pyplot as plt

# ==========================================
# 1. CONFIGURATION & PATHS
# ==========================================
CSV_ANNOTATIONS = '/Users/isabellesecord/Library/CloudStorage/OneDrive-ImperialCollegeLondon/AA - MRES PROJECT/Model Training Version 6/Call_Library/Saccopteryx Bilineata classification verified list.csv'
OUTPUT_DIR = '/Users/isabellesecord/Library/CloudStorage/OneDrive-ImperialCollegeLondon/AA - MRES PROJECT/Model Training Version 6/Call_Library/1 - positive target'

# Base Search Directories by Year/Volume / to mounted RDS volumes
SEARCH_BASE_PATHS = [
    "/Volumes/AcousticData/AcousticData_2024/AcousticData_Turmalina_2024/BatData_Turmalina_2024",
    "/Volumes/AcousticData/AcousticData_2023/Mombak_Turmalina_2023/BatData_Turmalina_2023",
    "/Volumes/AcousticData/AcousticData_2025/AcousticData_Turmalina_2025_Sample2/BatData_Turmalina_2025_Sample2",
    "/Volumes/AcousticData"
]

# sets target sample rate and clip duration for 0.5s audio segments
TARGET_SR = 256000
CLIP_DURATION = 0.5
TARGET_SAMPLES = int(TARGET_SR * CLIP_DURATION)  # 128,000 samples

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# 2. FAST FILE FINDER
# ==========================================
def find_audio_file_fast(filename, site_hint=None):
    """
    Directly locates audio files on network drives without freezing on deep walks.
    """
    filename = filename.strip()
    if not filename.lower().endswith('.wav'):
        filename += ".wav"

    # 1. Try direct site folder match if site is provided
    if site_hint and pd.notna(site_hint):
        site_str = str(site_hint).strip()
        for base in SEARCH_BASE_PATHS:
            # Check common folder patterns: base/SM_05/Data/file.wav or base/SM_05/file.wav
            candidate1 = os.path.join(base, site_str, "Data", filename)
            candidate2 = os.path.join(base, site_str, filename)
            if os.path.exists(candidate1): return candidate1
            if os.path.exists(candidate2): return candidate2

    # 2. Shallow search across top-level site folders (max depth 2), to search for in the songmeter folders that have nested data folders
    for base in SEARCH_BASE_PATHS:
        if os.path.exists(base):
            try:
                for entry in os.scandir(base):
                    if entry.is_dir():
                        p1 = os.path.join(entry.path, filename)
                        p2 = os.path.join(entry.path, "Data", filename)
                        if os.path.exists(p1): return p1
                        if os.path.exists(p2): return p2
            except PermissionError:
                continue

    return None

def find_all_energy_peaks(y, sr, clip_samples=128000, max_peaks=10, min_gap_samples=128000):
    """Locates top energy peaks ordered by amplitude, spaced at least 0.5s apart."""
    hop = 512
    rms = librosa.feature.rms(y=y, frame_length=2048, hop_length=hop)[0]
    
    rms_copy = rms.copy()
    peak_segments = []
    
    for _ in range(max_peaks):
        peak_frame = np.argmax(rms_copy)
        if rms_copy[peak_frame] < 1e-4:
            break
            
        peak_sample = peak_frame * hop
        half_clip = clip_samples // 2
        start_sample = max(0, peak_sample - half_clip)
        end_sample = start_sample + clip_samples
        
        if end_sample > len(y):
            end_sample = len(y)
            start_sample = max(0, end_sample - clip_samples)
            
        segment = y[start_sample:end_sample]
        if len(segment) < clip_samples:
            segment = np.pad(segment, (0, clip_samples - len(segment)), mode='constant')
            
        peak_segments.append(segment)
        
        frame_start = max(0, peak_frame - (min_gap_samples // hop))
        frame_end = min(len(rms_copy), peak_frame + (min_gap_samples // hop))
        rms_copy[frame_start:frame_end] = 0
        
    return peak_segments

def plot_spectrogram(segment, sr, species_name, original_filename, peak_idx, total_peaks):
    """Generates an interactive 256 Mel Bin Spectrogram."""
    plt.ion()
    plt.close('all')
    fig, ax = plt.subplots(figsize=(10, 4.5))
    
    S = librosa.feature.melspectrogram(
        y=segment, sr=sr, n_fft=2048, hop_length=512, 
        n_mels=256, fmin=20000, fmax=100000
    )
    S_db = librosa.power_to_db(S, ref=np.max)
    
    img = librosa.display.specshow(
        S_db, sr=sr, hop_length=512, x_axis='time', y_axis='mel',
        fmin=20000, fmax=100000, cmap='magma', ax=ax
    )
    
    fig.colorbar(img, ax=ax, format='%+2.0f dB')
    ax.set_title(
        f"Species: {species_name} | File: {original_filename}\n"
        f"Peak {peak_idx + 1} of {total_peaks} (Highest RMS Window)", 
        fontsize=11, fontweight='bold'
    )
    plt.tight_layout()
    plt.show()
    plt.pause(0.1)

# ==========================================
# 3. MAIN INTERACTIVE LOOP
# ==========================================
def main():
    if not os.path.exists(CSV_ANNOTATIONS):
        print(f"❌ CSV File not found at: {CSV_ANNOTATIONS}")
        sys.exit(1)
        
    df = pd.read_csv(CSV_ANNOTATIONS)
    df = df.drop_duplicates().reset_index(drop=True)
    
    col_file = df.columns[0]     # 'Filepath' (contains WAV filename)
    col_species = df.columns[1]  # 'Species'
    col_site = df.columns[2] if len(df.columns) > 2 else None  # 'Site'
    
    print(f"📋 Loaded {len(df)} unique entries from CSV.")
    print("⌨️  Interactive Commands: [y] Save | [n] Next Peak | [s] Skip File | [p] Pause | [q] Quit\n")
    
    saved_files = set(os.listdir(OUTPUT_DIR))
    
    for idx, row in df.iterrows():
        fname = str(row[col_file]).strip()
        species = str(row[col_species]).strip().replace(" ", "_")
        site_val = row[col_site] if col_site else None
        species_clean = re.sub(r'[^\w\-]', '_', species)
        
        if not fname.lower().endswith('.wav'):
            fname += ".wav"
            
        base_no_ext = fname.rsplit('.', 1)[0]
        
        # Check duplicate
        existing_matches = [f for f in saved_files if base_no_ext in f]
        if existing_matches:
            print(f"⏭️  [{idx+1}/{len(df)}] Skipping duplicate already processed: {fname}")
            continue
            
        print(f"\n────────────────────────────────────────────────────────")
        print(f"📌 [{idx+1}/{len(df)}] Searching for: {fname} (Site: {site_val}) | Label: {species_clean}")
        
        full_path = find_audio_file_fast(fname, site_val)
        if not full_path:
            print(f"⚠️ Could not find '{fname}' on network volume. Skipping.")
            continue
            
        print(f"⚡ Located at: {full_path}")
        
        try:
            y, native_sr = sf.read(full_path)
            if native_sr != TARGET_SR:
                y = librosa.resample(y, orig_sr=native_sr, target_sr=TARGET_SR)
                
            peaks = find_all_energy_peaks(y, TARGET_SR, TARGET_SAMPLES, max_peaks=10)
            if not peaks:
                print("⚠️ File is completely silent or corrupted. Skipping.")
                continue
                
            peak_idx = 0
            while peak_idx < len(peaks):
                segment = peaks[peak_idx]
                plot_spectrogram(segment, TARGET_SR, species_clean, fname, peak_idx, len(peaks))
                
                cmd = input("👉 Command [y=Save, n=Next Peak, s=Skip, p=Pause, q=Quit]: ").strip().lower()
                
                if cmd == 'y' or cmd == '':
                    out_fname = f"{base_no_ext}_peak{peak_idx+1}_{species_clean}.wav"
                    out_path = os.path.join(OUTPUT_DIR, out_fname)
                    sf.write(out_path, segment, TARGET_SR)
                    saved_files.add(out_fname)
                    print(f"💾 SAVED CLIP: {out_fname}")
                    plt.close('all')
                    break
                    
                elif cmd == 'n':
                    peak_idx += 1
                    if peak_idx >= len(peaks):
                        print("⚠️ Reached last energy peak in this file. Resetting to peak 1.")
                        peak_idx = 0
                    continue
                    
                elif cmd == 's':
                    print(f"⏭️  Skipped {fname}")
                    plt.close('all')
                    break
                    
                elif cmd == 'p':
                    input("⏸️  PAUSED. Press [Enter] to resume...")
                    continue
                    
                elif cmd == 'q':
                    print("\n🛑 Quitting clipper. Progress saved!")
                    plt.close('all')
                    sys.exit(0)
                else:
                    print("❓ Unknown command. Use 'y', 'n', 's', 'p', or 'q'.")
                    
        except Exception as e:
            print(f"❌ Error processing {fname}: {e}")

if __name__ == "__main__":
    main()